# ⚡ Consultas Analíticas Avanzadas OLAP y Reporting Ejecutivo

### Autor: Néstor León | Business Intelligence & Analytics
---

Un almacén de datos no cobra vida hasta que extraemos respuestas estratégicas:
- *¿Cuáles son los productos y regiones que concentran el 80% de las ventas?*
- *¿Cuál fue el crecimiento intermensual (MoM) de cada sucursal?*
- *¿Cómo calculamos subtotales jerárquicos automáticos sin lanzar 10 consultas separadas?*

En este notebook conectamos con el Data Warehouse creado en el notebook anterior y resolvemos estas preguntas mediante **SQL analítico avanzado**:
1. Operadores multidimensionales: `ROLLUP` y `CUBE`.
2. Funciones de Ventana (*Window Functions*): `SUM() OVER`, `ROW_NUMBER`, `DENSE_RANK`.
3. Comparativas temporales dinámicas con `LAG()` y `LEAD()`.


In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Conectamos con el Data Warehouse
con = duckdb.connect('sales_datawarehouse.duckdb')
print("Conexión activa con el Data Warehouse.")


## 1. Agregación Jerárquica Multidimensional con ROLLUP

`ROLLUP` calcula en una única pasada los niveles agregados: 
- Total por Tienda
- Subtotal por División
- Subtotal por Región
- Gran Total de la compañía


In [ ]:
query_rollup = """
SELECT 
    COALESCE(s.region, '--- GRAN TOTAL ---') AS region,
    COALESCE(s.division_name, '--- TOTAL DIVISION ---') AS division,
    COALESCE(s.store_name, '--- TOTAL TIENDA ---') AS tienda,
    COUNT(f.sales_id) AS transacciones,
    ROUND(SUM(f.total_sales_amount), 2) AS facturacion_total
FROM fact_sales f
JOIN dim_store s ON f.store_key = s.store_key
GROUP BY ROLLUP (s.region, s.division_name, s.store_name)
ORDER BY s.region NULLS LAST, facturacion_total DESC;
"""

df_rollup = con.execute(query_rollup).df()
df_rollup.head(10)


## 2. Análisis Temporal y Tasa de Variación Intermensual (MoM)

Utilizamos la función de ventana `LAG()` para traer el valor de facturación del mes inmediatamente anterior sin necesidad de hacer un auto-join complejo.


In [ ]:
query_mom = """
WITH ventas_mensuales AS (
    SELECT 
        t.year_number,
        t.month_number,
        t.month_name,
        SUM(f.total_sales_amount) AS ventas_mes
    FROM fact_sales f
    JOIN dim_time t ON f.time_key = t.time_key
    GROUP BY t.year_number, t.month_number, t.month_name
)
SELECT 
    year_number,
    month_number,
    month_name,
    ROUND(ventas_mes, 2) AS facturacion,
    ROUND(LAG(ventas_mes, 1) OVER (ORDER BY year_number, month_number), 2) AS facturacion_mes_ant,
    ROUND(
        (ventas_mes - LAG(ventas_mes, 1) OVER (ORDER BY year_number, month_number)) 
        / NULLIF(LAG(ventas_mes, 1) OVER (ORDER BY year_number, month_number), 0) * 100, 2
    ) AS variacion_pct_mom
FROM ventas_mensuales
ORDER BY year_number, month_number;
"""

df_mom = con.execute(query_mom).df()
df_mom


### Visualización del Ritmo de Ventas Mensuales


In [ ]:
plt.figure(figsize=(11, 4))
plt.bar(df_mom['month_name'], df_mom['facturacion'], color='#3498db', alpha=0.85)
plt.title("Facturación Total Mensual del Negocio", fontsize=13, fontweight='bold')
plt.xlabel("Mes")
plt.ylabel("Facturación (€)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## 3. Análisis de Pareto: Ranking y Contribución Acumulada de Clientes

Identificamos los clientes estrella y su peso porcentual acumulado sobre la facturación global mediante `SUM() OVER ()`.


In [ ]:
query_pareto = """
WITH ranking_clientes AS (
    SELECT 
        c.customer_name,
        c.city,
        SUM(f.total_sales_amount) AS facturacion_cliente,
        DENSE_RANK() OVER (ORDER BY SUM(f.total_sales_amount) DESC) AS rank_posicion
    FROM fact_sales f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    GROUP BY c.customer_name, c.city
)
SELECT 
    rank_posicion,
    customer_name,
    city,
    ROUND(facturacion_cliente, 2) AS facturacion,
    ROUND(SUM(facturacion_cliente) OVER (ORDER BY rank_posicion) / SUM(facturacion_cliente) OVER () * 100, 2) AS pct_acumulado_total
FROM ranking_clientes
ORDER BY rank_posicion;
"""

df_pareto = con.execute(query_pareto).df()
df_pareto


## 4. Conclusiones y Valor Estratégico

1. **Visibilidad Ejecutiva Inmediata**: Con apenas tres consultas OLAP estructuradas, hemos proporcionado a la dirección general la foto completa: qué regiones tiran del negocio, cómo evoluciona la facturación mes a mes y qué cuentas clave suponen el núcleo del margen comercial.
2. **Puente Natural con BI**: Estos datamarts resultantes están en el formato exacto requerido por herramientas como Power BI o Tableau para crear cuadros de mando interactivos sin penalizar las bases de datos transaccionales operativas.
